In [2]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Successive OBR forecasts of underlying debt vs outturn, % of GDP. Source: OBR EFO Oct 2026, Chart 7.2.
xl = "../Data/efo-october-2026-charts-and-tables-chapter-7.xlsx"   # adjust path
raw = pd.read_excel(xl, sheet_name="C7.2", header=None)

order = ["November 2022", "November 2023", "October 2024",
         "November 2025", "October 2026", "Outturn"]

# right-hand block of the sheet (F26:L47): header on row 26, data rows 27-47
d = raw.iloc[26:47, 5:12].copy()
d.columns = ["fy", "Outturn", "November 2022", "November 2023",
             "October 2024", "November 2025", "October 2026"]
d["year"] = d["fy"].astype(str).str[:4].astype(int)          # "2026-27" -> 2026
for c in order:
    d[c] = pd.to_numeric(d[c], errors="coerce")

# sanity check against Table A.9 (PSND ex BoE, % of GDP, Oct 2026 forecast)
latest = d.set_index("fy")["October 2026"]
for fy, v in {"2026-27": 92.4, "2027-28": 93.8, "2028-29": 94.7,
              "2029-30": 94.3, "2030-31": 94.2, "2031-32": 94.0}.items():
    assert abs(latest[fy] - v) < 0.1, (fy, latest[fy], v)

long = d.melt(id_vars=["fy", "year"], value_vars=order,
              var_name="series", value_name="value").dropna(subset=["value"])
long["kind"] = np.where(long["series"] == "Outturn", "Outturn", "Forecast")

base = alt.Chart(long).encode(
    detail="series:N",
    color=alt.Color("kind:N", sort=["Forecast", "Outturn"],
        scale=alt.Scale(domain=["Forecast", "Outturn"], range=["#36B7B4", "#E6224B"]),
        legend=None))

lines = base.mark_line().encode(
    x=alt.X("year:Q", title=None, scale=alt.Scale(domain=[2011, 2032]),
        axis=alt.Axis(values=[2011, 2016, 2021, 2026, 2031],
            labelExpr="datum.value + '-' + slice(toString(datum.value + 1), 2)")),
    y=alt.Y("value:Q", title="Debt (% of GDP)", scale=alt.Scale(zero=False)),
    # ordinal scales: the default point scale only reads the first two range values
    opacity=alt.Opacity("series:N", legend=None,
        scale=alt.Scale(type="ordinal", domain=order, range=[0.35, 0.5, 0.65, 0.8, 1, 1])),
    strokeWidth=alt.StrokeWidth("kind:N", legend=None,
        scale=alt.Scale(type="ordinal", domain=["Forecast", "Outturn"], range=[1.8, 3.2])))

# end-of-line direct labels, nudged (pp of GDP) so neighbours don't collide
nudge = {"November 2022": -0.35, "November 2023": 0.3, "November 2025": 0.6, "October 2026": -0.6}
nudge_expr = "datum.value + (" + " : ".join(
    f"datum.series == '{k}' ? {v}" for k, v in nudge.items()) + " : 0)"

labels = base.transform_calculate(ypos=nudge_expr).mark_text(
    align="left", dx=5, fontSize=11).encode(
    x=alt.X("year:Q", aggregate="max"),
    y=alt.Y("ypos:Q", aggregate={"argmax": "year"}, title="Debt (% of GDP)"),
    text="series:N")

# hover: vertical rule, tooltip lists every series for that year
nearest = alt.selection_point(fields=["year"], nearest=True, on="mouseover",
                              empty=False, clear="mouseout")
selectors = alt.Chart(long).mark_point(opacity=0).encode(x="year:Q").add_params(nearest)
rule = alt.Chart(long).transform_pivot("series", value="value", groupby=["year", "fy"]).mark_rule(
    color="#c9c9c9").encode(
    x="year:Q",
    tooltip=[alt.Tooltip("fy:N", title="Year")] +
            [alt.Tooltip(field=s, type="quantitative", format=".1f", title=s) for s in order]
).transform_filter(nearest)

caption = alt.Title(
    text="Source: OBR, Economic and fiscal outlook, October 2026 (Chart 7.2)",
    subtitle=["Underlying public sector net debt (excluding the Bank of England): successive forecasts vs outturn. Older forecasts are fainter."],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (lines + labels + selectors + rule).properties(
    width=700, height=300, title=caption).configure(
    background="white", font="Circular Std").configure_view(
    fill="transparent", stroke="transparent")

styles.save(chart, path=".", name="debt_forecasts_vs_outturn", svg=True)
chart.save("debt_forecasts_vs_outturn.png", scale_factor=2.0)
chart.save("debt_forecasts_vs_outturn.html")          # standalone interactive file
chart

alt.LayerChart(...)